# TASK 3 — Cleaning Data

**Objective:** Demonstrate professional-level data cleaning skills by transforming the deliberately messy café sales dataset into a clean, analysis-ready dataset.

### Cleaning decisions
- Treat `ERROR`, `UNKNOWN`, blank strings, and nulls as missing values where appropriate.
- Numeric columns: convert to numeric and use **median imputation**.
- Categorical columns: use **mode imputation**.
- `Transaction Date`: convert to `datetime` and use **forward fill**, then backward fill if a missing value occurs at the beginning.
- Remove duplicate rows and document the number removed.
- Standardize categorical text and date formatting.
- Use the **IQR method** to detect numeric outliers.
- Keep genuine sales outliers rather than deleting them automatically; document them.
- Keep `Transaction ID` as a string.
- Ensure monetary fields are `float`.


In [ ]:
# 1. Imports
import pandas as pd
import numpy as np
from google.colab import files
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")


## 2. Load the dataset

Upload `dirty_cafe_sales.csv` when prompted.


In [ ]:
uploaded = files.upload()

# Use the uploaded CSV file
csv_files = [name for name in uploaded.keys() if name.lower().endswith(".csv")]
if not csv_files:
    raise FileNotFoundError("Please upload a CSV file.")

file_name = csv_files[0]
df = pd.read_csv(file_name)

print(f"Loaded: {file_name}")
print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
display(df.head())


## 3. Initial data inspection

In [ ]:
print("Dataset shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

print("\nOriginal data types:")
display(df.dtypes.to_frame("dtype"))

print("\nFirst 5 rows:")
display(df.head())

print("\nBasic information:")
df.info()


## 4. Data quality report

The report checks:
1. Null values
2. Duplicate rows
3. Data-type issues
4. `ERROR` / `UNKNOWN` values
5. Numeric range anomalies
6. Invalid dates


In [ ]:
# Count explicit missing values
null_counts = df.isna().sum()

# Count placeholder values that should be treated as missing
placeholder_counts = {}
for col in df.columns:
    s = df[col].astype("string").str.strip().str.upper()
    placeholder_counts[col] = int(s.isin(["ERROR", "UNKNOWN"]).sum())

quality_report = pd.DataFrame({
    "Null Values": null_counts,
    "ERROR/UNKNOWN": pd.Series(placeholder_counts),
    "Unique Values": df.nunique(dropna=True),
    "Data Type": df.dtypes.astype(str)
})

quality_report["Total Missing-Like"] = (
    quality_report["Null Values"] + quality_report["ERROR/UNKNOWN"]
)

quality_report["Missing-Like %"] = (
    quality_report["Total Missing-Like"] / len(df) * 100
).round(2)

print("Duplicate rows:", df.duplicated().sum())
display(quality_report.sort_values("Total Missing-Like", ascending=False))


### 4.1 Data-type and value-range checks

`Quantity` should be between 1 and 5, prices should be positive, and total spending should be positive. The three numeric columns should also satisfy:

`Total Spent = Quantity × Price Per Unit`

These checks are performed **before** imputation so the report describes the original data quality.


In [ ]:
# Convert numeric columns temporarily only for validation
numeric_check = df.copy()

for col in ["Quantity", "Price Per Unit", "Total Spent"]:
    numeric_check[col] = pd.to_numeric(numeric_check[col], errors="coerce")

date_check = pd.to_datetime(df["Transaction Date"], errors="coerce")

range_anomalies = {
    "Quantity outside 1–5": int(
        ((numeric_check["Quantity"] < 1) | (numeric_check["Quantity"] > 5)).sum()
    ),
    "Price Per Unit <= 0": int(
        (numeric_check["Price Per Unit"] <= 0).sum()
    ),
    "Total Spent <= 0": int(
        (numeric_check["Total Spent"] <= 0).sum()
    ),
    "Invalid/missing dates": int(date_check.isna().sum())
}

# Arithmetic consistency only where all 3 numeric fields are present
complete_numeric = numeric_check[["Quantity", "Price Per Unit", "Total Spent"]].notna().all(axis=1)
inconsistent_total = (
    (numeric_check["Quantity"] * numeric_check["Price Per Unit"]
     - numeric_check["Total Spent"]).abs() > 1e-9
) & complete_numeric

range_report = pd.Series(range_anomalies, name="Count")
range_report.loc["Total Spent != Quantity × Price"] = int(inconsistent_total.sum())

display(range_report.to_frame())


## 5. Missing-data handling

### Strategy justification
- **Numeric columns (`Quantity`, `Price Per Unit`, `Total Spent`) → median imputation:** robust to skew and possible outliers.
- **Categorical columns (`Item`, `Payment Method`, `Location`) → mode imputation:** preserves the most common valid category without inventing a new category.
- **Transaction Date → forward fill, then backward fill:** maintains the chronological structure of the transaction data. This is preferable to replacing dates with a statistical mean.
- `ERROR` and `UNKNOWN` are first converted to real missing values.


In [ ]:
# Work on a copy
clean_df = df.copy()

# Strip whitespace from all string columns
for col in clean_df.select_dtypes(include="object").columns:
    clean_df[col] = clean_df[col].astype("string").str.strip()

# Convert placeholder values to missing
clean_df = clean_df.replace({
    "ERROR": np.nan,
    "UNKNOWN": np.nan,
    "error": np.nan,
    "unknown": np.nan,
    "": np.nan,
    " ": np.nan
})

# Correct data types
clean_df["Transaction ID"] = clean_df["Transaction ID"].astype("string")

for col in ["Quantity", "Price Per Unit", "Total Spent"]:
    clean_df[col] = pd.to_numeric(clean_df[col], errors="coerce")

clean_df["Transaction Date"] = pd.to_datetime(
    clean_df["Transaction Date"], errors="coerce"
)

# Categorical columns
categorical_cols = ["Item", "Payment Method", "Location"]

# Standardize category spelling/capitalization
for col in categorical_cols:
    clean_df[col] = clean_df[col].astype("string").str.strip()

# Fill categorical missing values with mode
for col in categorical_cols:
    mode_value = clean_df[col].mode(dropna=True)
    if len(mode_value) > 0:
        clean_df[col] = clean_df[col].fillna(mode_value.iloc[0])

# Fill numeric missing values with median
numeric_cols = ["Quantity", "Price Per Unit", "Total Spent"]
for col in numeric_cols:
    clean_df[col] = clean_df[col].fillna(clean_df[col].median())

# Fill dates chronologically
clean_df = clean_df.sort_values("Transaction Date", na_position="first")
clean_df["Transaction Date"] = clean_df["Transaction Date"].ffill().bfill()

# Make sure monetary values are float
clean_df["Price Per Unit"] = clean_df["Price Per Unit"].astype(float)
clean_df["Total Spent"] = clean_df["Total Spent"].astype(float)

# Quantity is numeric; keep it as integer after median imputation
clean_df["Quantity"] = clean_df["Quantity"].round().astype("int64")

print("Missing values after imputation:")
display(clean_df.isna().sum().to_frame("Missing Values"))


## 6. Duplicate removal

Duplicate rows are removed using all columns. The number removed is documented below.


In [ ]:
rows_before_duplicates = len(clean_df)

clean_df = clean_df.drop_duplicates().reset_index(drop=True)

duplicates_removed = rows_before_duplicates - len(clean_df)

print(f"Rows before duplicate removal: {rows_before_duplicates:,}")
print(f"Duplicate rows removed: {duplicates_removed:,}")
print(f"Rows after duplicate removal: {len(clean_df):,}")


## 7. Standardization

Examples of standardization:
- Text is stripped of leading/trailing whitespace.
- Placeholder values are converted to `NaN` before imputation.
- Categories are kept in a consistent format.
- Dates are stored as `datetime64`.
- IDs remain strings.
- Monetary columns are floats.


In [ ]:
# Optional explicit canonical mappings for known café categories
item_map = {
    "coffee": "Coffee", "tea": "Tea", "cake": "Cake", "cookie": "Cookie",
    "sandwich": "Sandwich", "salad": "Salad", "smoothie": "Smoothie",
    "juice": "Juice"
}

payment_map = {
    "cash": "Cash",
    "credit card": "Credit Card",
    "digital wallet": "Digital Wallet"
}

location_map = {
    "in-store": "In-store",
    "takeaway": "Takeaway"
}

clean_df["Item"] = clean_df["Item"].str.lower().map(item_map).fillna(clean_df["Item"])
clean_df["Payment Method"] = (
    clean_df["Payment Method"].str.lower().map(payment_map).fillna(clean_df["Payment Method"])
)
clean_df["Location"] = (
    clean_df["Location"].str.lower().map(location_map).fillna(clean_df["Location"])
)

print("Standardized categories:")
for col in categorical_cols:
    print(f"\n{col}:")
    print(sorted(clean_df[col].dropna().unique().tolist()))


## 8. Outlier detection using the IQR method

For each numeric column:

- Q1 = 25th percentile
- Q3 = 75th percentile
- IQR = Q3 − Q1
- Lower bound = Q1 − 1.5 × IQR
- Upper bound = Q3 + 1.5 × IQR

**Decision:** retain detected outliers because extreme café transactions can be legitimate observations. We document them instead of deleting them automatically.


In [ ]:
def iqr_outlier_report(data, columns):
    results = []

    for col in columns:
        q1 = data[col].quantile(0.25)
        q3 = data[col].quantile(0.75)
        iqr = q3 - q1
        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr

        mask = (data[col] < lower) | (data[col] > upper)

        results.append({
            "Column": col,
            "Q1": q1,
            "Q3": q3,
            "IQR": iqr,
            "Lower Bound": lower,
            "Upper Bound": upper,
            "Outlier Count": int(mask.sum()),
            "Outlier %": round(mask.mean() * 100, 2),
            "Decision": "Retain — potentially valid sales values"
        })

    return pd.DataFrame(results)

outlier_report = iqr_outlier_report(clean_df, numeric_cols)
display(outlier_report)


## 9. Final validation

The final dataset must have:
- no missing values,
- no duplicate rows,
- correct data types,
- valid numeric ranges,
- valid dates,
- consistent total spending.


In [ ]:
# Final validation
validation = {
    "Rows": len(clean_df),
    "Columns": len(clean_df.columns),
    "Missing cells": int(clean_df.isna().sum().sum()),
    "Duplicate rows": int(clean_df.duplicated().sum()),
    "Quantity outside 1–5": int(((clean_df["Quantity"] < 1) | (clean_df["Quantity"] > 5)).sum()),
    "Price <= 0": int((clean_df["Price Per Unit"] <= 0).sum()),
    "Total Spent <= 0": int((clean_df["Total Spent"] <= 0).sum()),
    "Invalid dates": int(clean_df["Transaction Date"].isna().sum()),
    "Total calculation mismatches": int(
        (clean_df["Total Spent"] -
         clean_df["Quantity"] * clean_df["Price Per Unit"]).abs().gt(1e-9).sum()
    )
}

display(pd.Series(validation, name="Result").to_frame())

print("\nFinal data types:")
display(clean_df.dtypes.to_frame("dtype"))

print("\nFinal sample:")
display(clean_df.head(10))


## 10. Export the cleaned dataset and reports

The notebook creates:
- `clean_cafe_sales.csv` — analysis-ready dataset
- `data_quality_report.csv` — initial data quality report
- `outlier_report.csv` — IQR outlier analysis


In [ ]:
# Export files
clean_output = "clean_cafe_sales.csv"
quality_output = "data_quality_report.csv"
outlier_output = "outlier_report.csv"

clean_df.to_csv(clean_output, index=False)
quality_report.to_csv(quality_output)
outlier_report.to_csv(outlier_output, index=False)

print("Files created successfully:")
print(f"1. {clean_output}")
print(f"2. {quality_output}")
print(f"3. {outlier_output}")

# Uncomment these lines in Colab if you want automatic downloads:
# files.download(clean_output)
# files.download(quality_output)
# files.download(outlier_output)


## 11. Conclusion

The dataset has been transformed into an analysis-ready format by:

1. Auditing missing values, duplicates, data types, placeholders, and range anomalies.
2. Converting `ERROR` and `UNKNOWN` to missing values.
3. Imputing numeric fields with medians and categorical fields with modes.
4. Filling transaction dates using forward fill/backward fill.
5. Removing duplicate rows and documenting the count.
6. Standardizing categorical values and date formatting.
7. Detecting numeric outliers using IQR and retaining them because they may represent legitimate transactions.
8. Validating the final dataset and exporting the cleaned CSV.
